[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/parallel_questions/parallel_questions.ipynb)

# Ask everything in one request

A briefing over a policy is a dozen narrow questions, not one essay. Asked separately, each call re-encodes the text. Asked together, the questions sit next to one copy of the text and do not attend to each other, so an extra question does not change the others.

This notebook times one request against the same questions one by one, and checks that the decisions match. `token_counts` shows the encoder work without running the model: `naive` is one sequence per option, `packed` is the packed rows, `cached` is the state once plus the question rows.

The state should stay inside the trained range. The default cut is 384 tokens; `usage.state_truncated` says when the tail was dropped.

Patterns: [Parallel questions](https://docs.typesafe.ai/cookbooks/parallel_questions) and [Speculative fan-out](https://docs.typesafe.ai/patterns/fan-out).


In [1]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


Twelve questions about one short policy. A choice has to match exactly. A score or a yes/no probability may drift by 0.05: that is the size of the int8 ONNX gap between the two shapes.


In [3]:
import time

POLICY = (
    "Магазин «Северная полка» продаёт книги и посуду. Товар надлежащего качества можно вернуть "
    "в течение 14 дней с даты получения, если сохранены товарный вид, упаковка и чек. "
    "Книги с повреждённой обложкой и посуда со сколом обмену не подлежат: это след использования. "
    "Если товар сломался сам в первые семь дней, магазин меняет его или возвращает деньги, "
    "доставку оплачивает магазин. Заявление пишут на адрес support@severnaya-polka.example, "
    "в теме указывают номер заказа. Деньги возвращают на ту же карту в срок до десяти рабочих дней. "
    "Персональные данные из заявления используются только для возврата и не передаются третьим лицам. "
    "Акции и скидки на этот порядок не влияют: срок 14 дней считается и для товара, купленного со скидкой."
)
QUESTIONS = {
    "kind": {"type": "choice", "instructions": "Что это за текст?",
             "criteria": {"policy": "правила возврата магазина", "ad": "реклама товара",
                          "news": "новость", "manual": "инструкция к прибору"}},
    "audience": {"type": "choice", "instructions": "Для кого написан текст?",
                 "criteria": {"buyer": "покупатель", "courier": "курьер",
                              "accountant": "бухгалтер", "child": "ребёнок"}},
    "contact": {"type": "choice", "instructions": "Как с магазином связаться по этому тексту?",
                "criteria": {"email": "по электронной почте", "phone": "по телефону",
                             "office": "прийти в офис", "chat": "через чат на сайте"}},
    "broken": {"type": "choice", "instructions": "Что магазин делает, если товар сломался сам в первую неделю?",
               "criteria": {"replace": "меняет или возвращает деньги", "repair": "только чинит",
                            "refuse": "отказывает", "credit": "даёт баллы на следующую покупку"}},
    "formality": {"type": "score", "instructions": "Насколько официален тон текста?",
                  "criteria": ["Совсем разговорный.", "Скорее разговорный.", "Нейтральный.",
                               "Скорее официальный.", "Канцелярский."]},
    "strictness": {"type": "score", "instructions": "Насколько жёсткие условия возврата?",
                   "criteria": ["Вернуть можно почти всё и почти всегда.",
                                "Условия мягкие, исключений мало.",
                                "Есть и срок, и заметные исключения.",
                                "Условия жёсткие, исключений много.",
                                "Вернуть практически нельзя."]},
    "refund": {"type": "noul", "instructions": "В тексте есть возврат денег или товара?"},
    "days14": {"type": "noul", "instructions": "Назван срок в 14 дней?"},
    "personal": {"type": "noul", "instructions": "Текст говорит, как поступят с персональными данными?"},
    "store_pays": {"type": "noul", "instructions": "Доставку при обмене брака оплачивает магазин?"},
    "sells_books": {"type": "noul", "instructions": "Магазин продаёт книги?"},
    "discount_blocks": {"type": "noul", "instructions": "Скидка отменяет право на возврат?"},
}
FLOAT_TOLERANCE = 0.05


def value_of(answer):
    if answer["type"] == "choice":
        return answer["choice"]
    if answer["type"] == "score":
        return answer["score"]
    return answer["noul"]


def same(left, right):
    if isinstance(left, float) and isinstance(right, float):
        return abs(left - right) <= FLOAT_TOLERANCE
    return left == right


The separate calls run first, on a cold state cache, so the packed call cannot warm the cache for them. `token_counts` is the structural comparison: `naive` re-encodes the text once per option, `packed` keeps one copy per row.


In [4]:
def run(judge):
    full = {"state": POLICY, "questions": QUESTIONS}
    print("token_counts", judge.token_counts(full))

    separate = {}
    started = time.perf_counter()
    for key, question in QUESTIONS.items():
        separate[key] = judge({"state": POLICY, "questions": {key: question}})
    elapsed_separate = time.perf_counter() - started

    started = time.perf_counter()
    together = judge(full)
    elapsed_together = time.perf_counter() - started

    widest = 0.0
    mismatches = []
    for key in QUESTIONS:
        left = value_of(together["answers"][key])
        right = value_of(separate[key]["answers"][key])
        if isinstance(left, float) and isinstance(right, float):
            widest = max(widest, abs(left - right))
        if not same(left, right):
            mismatches.append(key)
        print(f"  {key:16} together {left!s:20} apart {right}")

    separate_ms = 0.0
    for item in separate.values():
        separate_ms += item["usage"]["milliseconds"]
    print(f"\n12 requests, cold: {elapsed_separate:.2f}s  sum of milliseconds {separate_ms:.0f}")
    print(f"one request:       {elapsed_together:.2f}s  usage {together['usage']}")
    print("state_truncated", together["usage"]["state_truncated"],
          "state_tokens", together["usage"]["state_tokens"])
    print(f"max abs gap on numeric answers {widest:.4f}")
    if mismatches:
        print("mismatches", mismatches)
    else:
        print(f"all {len(QUESTIONS)} decisions agree")


In [5]:
run(judge)


token_counts {'candidates': 38, 'state_tokens': 158, 'naive': 8110, 'packed': 1312, 'packed_rows': 3, 'cached': 996, 'cached_rows': 3}
  kind             together policy               apart policy
  audience         together buyer                apart buyer
  contact          together email                apart email
  broken           together replace              apart replace
  formality        together 2.835736782629169    apart 2.835736579927092
  strictness       together 0.07137087005045473  apart 0.0713709132980798
  refund           together 0.9817473417729858   apart 0.9817473503176667
  days14           together 0.9857721580661715   apart 0.9857722048809535
  personal         together 0.4494728676609647   apart 0.44947251368520474
  store_pays       together 0.71656631539696     apart 0.7165661701291903
  sells_books      together 0.9549548280666442   apart 0.9549547972991186
  discount_blocks  together 0.4583165596480676   apart 0.4583167372192439

12 requests, cold: 10.53s